# Vision → MIDI (Sheet Music OMR) — Phase 2

**Dataset required:** PrIMuS — 87,678 sheet music PNG + agnostic encoding pairs.
Download from: https://grfia.dlsi.ua.es/primus/ (~2 GB compressed)

**Architecture:**
```
Sheet music PNG
  → VisionEncoder (ViT-style, 16×16 patches)
  → encoder memory (B, num_patches, d_model)
  → cross-attention in MusicTransformer decoder
  → MIDI event tokens → decode_midi() → .mid
```

Prerequisites: run `00_setup_and_data.ipynb` first.

In [ ]:
!pip install -q timm pretty-midi mido torchvision

In [ ]:
import os, torch
import matplotlib.pyplot as plt
from pathlib import Path

DRIVE    = '/content/drive/MyDrive/deep-techno-data'
PRIMUS   = os.path.join(DRIVE, 'primus')           # upload PrIMuS here once
CKPT_DIR = os.path.join(DRIVE, 'checkpoints', 'vision')
os.makedirs(CKPT_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## Dataset — PrIMuS

Manual step: download from https://grfia.dlsi.ua.es/primus/ and upload to `MyDrive/deep-techno-data/primus/`.

In [ ]:
primus_path = Path(PRIMUS)
if primus_path.exists():
    samples = list(primus_path.glob('**/*.png'))
    print(f'PrIMuS: {len(samples):,} PNG files found')
else:
    print('PrIMuS not found. Upload to:', PRIMUS)
    print('Download: https://grfia.dlsi.ua.es/primus/')

## Inspect a sample

In [ ]:
import glob
from PIL import Image

sample_pngs = sorted(glob.glob(os.path.join(PRIMUS, '**/*.png'), recursive=True))[:3]

if sample_pngs:
    fig, axes = plt.subplots(1, len(sample_pngs), figsize=(18, 4))
    for ax, path in zip(axes, sample_pngs):
        ax.imshow(Image.open(path).convert('RGB')); ax.axis('off')
        ax.set_title(Path(path).stem[:40], fontsize=8)
    plt.tight_layout(); plt.show()
else:
    print('No PNGs found — upload PrIMuS first.')

## VisionEncoder

In [ ]:
from deepTechno.encoders.vision_encoder import VisionEncoder

encoder = VisionEncoder(
    img_size    = (128, 512),   # H×W — crop/pad sheet music to this
    patch_size  = 16,
    d_model     = 512,
    in_channels = 1,            # grayscale
).to(device)

dummy = torch.zeros(2, 1, 128, 512).to(device)
out   = encoder(dummy)
print('Encoder output:', out.shape)   # (B, 256, 512)

## PrIMuS Dataset

Each sample: PNG + `.agnostic` music symbol annotations.
Targets are placeholder zeros until agnostic→MIDI conversion is implemented.

In [ ]:
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

IMG_SIZE = (128, 512)
MAX_SEQ  = 511   # input length; target is shifted by 1

class PrimusDataset(Dataset):
    def __init__(self, root, img_size=IMG_SIZE, max_seq=MAX_SEQ):
        self.samples   = sorted(Path(root).glob('**/*.png'))
        self.max_seq   = max_seq
        self.transform = transforms.Compose([
            transforms.Grayscale(),
            transforms.Resize(img_size),
            transforms.ToTensor(),
        ])

    def __len__(self):  return len(self.samples)

    def __getitem__(self, idx):
        img    = self.transform(Image.open(self.samples[idx]).convert('RGB'))
        target = torch.zeros(self.max_seq + 1, dtype=torch.long)  # placeholder
        return img, target

if primus_path.exists():
    ds = PrimusDataset(PRIMUS)
    print(f'Dataset: {len(ds)} samples')
    img, tgt = ds[0]
    print(f'Image: {img.shape}, target: {tgt.shape}')

## Model — VisionEncoder + MusicTransformer

Training strategy:
1. Freeze VisionEncoder for first `FREEZE_EPOCHS` (train decoder only)
2. Fine-tune end-to-end

In [ ]:
from deepTechno.model.transformer import MusicTransformer
from deepTechno.model.constants import VOCAB_SIZE

decoder = MusicTransformer(
    n_layers        = 6,
    num_heads       = 8,
    d_model         = 512,
    dim_feedforward = 2048,
    max_sequence    = MAX_SEQ + 1,
    rpr             = True,
).to(device)

print(f'Encoder: {sum(p.numel() for p in encoder.parameters())/1e6:.1f}M params')
print(f'Decoder: {sum(p.numel() for p in decoder.parameters())/1e6:.1f}M params')

## Training loop

In [ ]:
from deepTechno.model.loss import SmoothCrossEntropyLoss

EPOCHS        = 20
FREEZE_EPOCHS = 5
LR            = 1e-4
BATCH         = 8

criterion = SmoothCrossEntropyLoss()
optimizer = torch.optim.Adam(
    list(decoder.parameters()) + list(encoder.parameters()), lr=LR)

if primus_path.exists():
    loader = DataLoader(PrimusDataset(PRIMUS), batch_size=BATCH, shuffle=True, num_workers=2)
    losses = []

    for epoch in range(EPOCHS):
        for p in encoder.parameters():
            p.requires_grad = (epoch >= FREEZE_EPOCHS)

        epoch_loss = 0.0
        for imgs, targets in loader:
            imgs, targets = imgs.to(device), targets.to(device)
            enc_mem  = encoder(imgs)                     # (B, N_patches, d_model)
            tgt_in   = targets[:, :-1]
            tgt_out  = targets[:, 1:]
            logits   = decoder(tgt_in, encoder_memory=enc_mem)   # cross-attn to vision
            loss     = criterion(logits.reshape(-1, VOCAB_SIZE), tgt_out.reshape(-1))
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            epoch_loss += loss.item()

        avg = epoch_loss / len(loader)
        losses.append(avg)
        print(f'Epoch {epoch+1:02d}/{EPOCHS}  loss={avg:.4f}')
        torch.save({'encoder': encoder.state_dict(), 'decoder': decoder.state_dict()},
                   os.path.join(CKPT_DIR, f'vision_ep{epoch+1:02d}.pt'))

    plt.plot(losses); plt.title('Vision→MIDI loss')
    plt.xlabel('epoch'); plt.ylabel('loss'); plt.show()
else:
    print('Skipped — upload PrIMuS to', PRIMUS, 'first.')

## Generate MIDI from a sheet music image

In [ ]:
from deepTechno.data.tokenizer import decode_midi
from deepTechno.model.constants import TOKEN_END

_tf = transforms.Compose([
    transforms.Grayscale(), transforms.Resize(IMG_SIZE), transforms.ToTensor()])

def generate_from_image(image_path, temperature=1.0, max_len=512):
    encoder.eval(); decoder.eval()
    img = _tf(Image.open(image_path).convert('RGB')).unsqueeze(0).to(device)
    with torch.no_grad():
        mem    = encoder(img)
        x      = torch.zeros(1, 1, dtype=torch.long, device=device)
        tokens = []
        for _ in range(max_len):
            logits = decoder(x, encoder_memory=mem)[:, -1, :] / temperature
            tok    = torch.multinomial(torch.softmax(logits, -1), 1)
            if tok.item() == TOKEN_END:
                break
            tokens.append(tok.item())
            x = torch.cat([x, tok.unsqueeze(0)], dim=1)
    return decode_midi(tokens)

if sample_pngs:
    midi_obj = generate_from_image(sample_pngs[0])
    out_path = os.path.join(CKPT_DIR, 'generated_vision.mid')
    midi_obj.write(out_path)
    print('Saved:', out_path)

    import pretty_midi
    from IPython.display import Audio, display
    display(Audio(pretty_midi.PrettyMIDI(out_path).fluidsynth(), rate=44100))
else:
    print('No sample PNGs available.')